# Working with the API

## Introduction  

In this notebook, we will explore how to **use Anthropic's Python SDK to interact with Claude through the Messages API**. We will cover the basic setup, prompt construction, conversational context, and a few useful generation parameters.

The goal is to move from a minimal example to a realistic workflow: create a client, send a request, process the response, and keep a short conversation history.  

This notebook introduces the Anthropic Python SDK and shows how to send messages, manage conversation context, and control generation.

## Import the SDK and load credentials  

Before making API calls, we need to import the Anthropic client and load environment variables such as the API key. This is done with a `helper function` that reads values from a .env file (see `.env_example`).  

In [3]:
from anthropic import Anthropic

In [ ]:
# Load environment variables such as ANTHROPIC_API_KEY
from helper import load_env
load_env()

In [ ]:
# Initialize the Anthropic client
client = Anthropic()

## Select a model  

Anthropic exposes different Claude models. The model name is usually stored in a variable so it can be reused across notebook cells and experiments.

In [ ]:
# Example model configuration.
MODEL_NAME="claude-sonnet-5"  #MODEL_NAME="claude-3-5-sonnet-20241022"

## Making API call  

The `messages.create()` method is the core API call. It accepts:  

- a model name  
- a maximum token limit  
- a list of messages  
  
Each message includes a role and content.  

In [ ]:
# A basic prompt request: ask Claude to write a haiku
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write a haiku about Anthropic"}
    ]
)

print(response.content[0].text)

Here's a haiku about Anthropic:

Careful minds building
Bridges of trust, line by line—
Claude learns to listen.


In [6]:
response

Message(id='msg_011CfNVLrxqHvWp737K1ZwNp', content=[TextBlock(citations=None, text="Here's a haiku about Anthropic:\n\nCareful minds building\nBridges of trust, line by line—\nClaude learns to listen.", type='text')], model='claude-sonnet-5', role='assistant', stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=18, output_tokens=46, server_tool_use=None, service_tier='standard', output_tokens_details={'thinking_tokens': 0}, inference_geo='global'), container=None, stop_details=None, diagnostics=None)

**Explanation output** The response is a structured object containing metadata, such as the message ID, and content blocks. In this notebook, we print only the blocks that actually contain text. This is a safe pattern, because the SDK may return other block types depending on model behavior and API updates.

## The Messages Format  

Anthropic's Messages API uses a list of messages, where each message is a dictionary with:  

- `role`: either "user" or "assistant"  
- `content`: the text or structured content for that turn  

This format is fundamental for multi-turn conversations.

 This example shows how to keep conversation context across turns.

In [ ]:
# The assistant is told to respond in Dutch.
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Hello! Only speak to me in Dutch"},
        {"role": "assistant", "content": "Hallo!"},
        {"role": "user", "content": "How are you?"}
    ]
)


#print(response.content[0].text)
# The response object contains content blocks.
# In many current SDK versions, the text is available via block.text.
# A robust way to print the generated text:
for block in response.content:
    if hasattr(block, "text"):
        print(block.text)



Met mij gaat het goed, dank je! Hoe gaat het met jou?


This example shows how conversation context is preserved. Once the assistant has responded earlier, the next user message can be interpreted in context.

##  Building a simple chatbot  

A chatbot can be built by storing the conversation history in a list of messages. Each time the user enters a new message, it is appended to the list, and the model generates a reply based on the whole conversation so far.

In [ ]:
print("Simple Chatbot (type 'quit' to exit)")
# Store conversation history
messages = []
while True:
    # Get user input
    user_input = input("You: ")
    # Check for quit command
    if user_input.lower() == 'quit':
        print("Goodbye!")
        break
    # Add the user's latest message to the conversation history
    messages.append({"role": "user", "content": user_input})
    try:
        # Get response from Claude
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=200,
            messages=messages
        )
        # Extract and print Claude's response
        asst_message = response.content[0].text
        print("Assistant:", asst_message)
        
        # Store the assistant response so the model can continue the exchange
        messages.append({"role": "assistant", "content": asst_message})
        
    except Exception as e:
        print(f"An error occurred: {e}")

Simple Chatbot (type 'quit' to exit)
An error occurred: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': 'messages.0: user messages must have non-empty content'}, 'request_id': 'req_011CfNVVzHMvg8iMGGPK6qEJ'}
Assistant: # Overview of Anthropic's Models

Anthropic is an AI safety company that develops the **Claude** family of large language models. Here's a breakdown:

## Claude Model Generations

**Claude 1** (2023)
- Initial public release
- Established Anthropic's approach to helpful, harmless, honest AI

**Claude 2** (2023)
- Improved reasoning and coding capabilities
- Extended context window (up to 100K tokens)

**Claude 3 Family** (early 2024)
- Three tiers released together:
  - **Haiku** – fastest, most compact, cost-effective

An error occurred: 'ThinkingBlock' object has no attribute 'text'
An error occurred: 'ThinkingBlock' object has no attribute 'text'
An error occurred: 'ThinkingBlock' object has no attribute 'text'
An error occurr

This pattern is common in real-world chatbot applications, because it allows the model to remember prior turns and respond in a consistent style.

## Prefilling the Assistant Response  

In some situations, you may want to guide the assistant by providing a pre-written assistant message. This is useful when continuing a draft, shaping the tone, or maintaining a specific expected direction.

In [ ]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=1000,
    messages=[
        {"role": "user", "content": "Write a short poem about Dutch people"},
        {"role": "assistant", "content": "Oink"},
        {"role": "user", "content": "Continue the poem in a more polished, poetic style."}
    ]
)

#print(response.content[0].text)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

Low lands lifted by patient hands,
where dikes hold back the sea's demands,
and windmills turn on quiet air,
their wooden arms in slow prayer.

Canals like silver ribbons wind
through cities calm and clean and kind,
where bicycles outnumber cars,
and gabled roofs salute the stars.

Tall as their windmills, plain of speech,
with tulips blooming, bright, in reach—
a people practical and free,
who built their home up from the sea.


**Explanation output** This example shows that an assistant message does not always have to be generated by the model in the same call. It can be included explicitly to steer the next output.  

## Model Parameters    

The API supports several generation parameters. Some of the most important ones are:  

- `max_tokens`  
- `stop_sequences`  
- other model-specific generation controls  

These parameters help us guide the model's output length and shape.

### Max Tokens  

`max_tokens` limits the total length of the model's output. It is one of the simplest ways to keep responses concise or allow longer output when needed.

In [12]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Write me an essay on LLMs"},
    ]
)
print(response.content[0].text)

# Large Language Models: Transforming How We Interact with Information

Large Language Models (LLMs) represent one of the most significant technological developments of the early 21st century, fundamentally changing how humans interact with computers, access information, and even think about the nature of intelligence itself. These sophisticated artificial intelligence


In [13]:
response

Message(id='msg_011CfNW8LqPHeSYy2Ht9r15y', content=[TextBlock(citations=None, text='# Large Language Models: Transforming How We Interact with Information\n\nLarge Language Models (LLMs) represent one of the most significant technological developments of the early 21st century, fundamentally changing how humans interact with computers, access information, and even think about the nature of intelligence itself. These sophisticated artificial intelligence', type='text')], model='claude-sonnet-5', role='assistant', stop_reason='max_tokens', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=17, output_tokens=100, server_tool_use=None, service_tier='standard', output_tokens_details={'thinking_tokens': 0}, inference_geo='global'), container=None, stop_details=None, diagnostics=None)

### Stop Sequences  

`stop_sequences` tells the model to stop generating once it encounters a specific string. This is especially useful when we want structured output, such as a numbered list that should terminate at a certain point.

In [14]:
prompt = """
Generate a numbered, ordered list of technical topics 
I should learn if I want to work on LLMs
"""
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=500,
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

# Technical Topics for Working on LLMs

## Foundations
1. Linear algebra (vectors, matrices, eigenvalues/eigenvectors, matrix decomposition)
2. Probability and statistics (distributions, Bayesian thinking, hypothesis testing)
3. Calculus (derivatives, gradients, chain rule, partial derivatives)
4. Python programming (proficiency with NumPy, PyTorch/TensorFlow)
5. Basic optimization theory (convex optimization, gradient descent variants)

## Machine Learning Basics
6. Supervised/unsupervised learning fundamentals
7. Neural network basics (perceptrons, activation functions, backpropagation)
8. Loss functions and optimization (SGD, Adam, learning rate schedules)
9. Regularization techniques (dropout, weight decay, batch normalization)
10. Evaluation metrics (perplexity, BLEU, ROUGE, accuracy)

## Deep Learning Architectures
11. Feedforward neural networks
12. Convolutional Neural Networks (CNNs) — for foundational understanding
13. Recurrent Neural Networks (RNNs, LSTMs, GRUs)
14. Attenti

In the following cell, the model is asked to produce a numbered list, but generation stops as soon as the sequence "4." appears (`stop_sequences`= [4]). This is a practical pattern when output needs to be concise or bounded.

In [15]:
prompt = """
Generate a numbered, ordered list of technical topics 
I should learn if I want to work on LLMs
"""
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=500,
    stop_sequences=["4."],
    messages=[{"role": "user", "content": prompt}],
)
print(response.content[0].text)

# Technical Topics for Working on LLMs

## Foundations

1. **Linear Algebra** — vectors, matrices, eigenvalues, matrix decomposition
2. **Probability & Statistics** — distributions, Bayes' theorem, maximum likelihood estimation
3. **Calculus** — derivatives, gradients, chain rule (for backpropagation)



In [16]:
response

Message(id='msg_011CfNWBFk8d97VEFAerQTW6', content=[TextBlock(citations=None, text="# Technical Topics for Working on LLMs\n\n## Foundations\n\n1. **Linear Algebra** — vectors, matrices, eigenvalues, matrix decomposition\n2. **Probability & Statistics** — distributions, Bayes' theorem, maximum likelihood estimation\n3. **Calculus** — derivatives, gradients, chain rule (for backpropagation)\n", type='text')], model='claude-sonnet-5', role='assistant', stop_reason='stop_sequence', stop_sequence='4.', type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, input_tokens=40, output_tokens=127, server_tool_use=None, service_tier='standard', output_tokens_details={'thinking_tokens': 0}, inference_geo='global'), container=None, stop_details=None, diagnostics=None)

## Temperature and generation behavior  

Older tutorials often include a temperature parameter. However, for some modern Anthropic models, temperature may be deprecated or ignored by the model in favor of default generation behavior. This is an important reminder that the SDK and API evolve over time.

A robust notebook should therefore:

keep code compatible with current SDK behavior
avoid assuming every response block is a TextBlock
handle response content defensively

In [23]:
def demonstrate_temperature():
    temperatures = [0, 1]
    for temperature in temperatures:
        print(f"Prompting Claude three times with temperature of {temperature}")
        print("================")
        for i in range(3):
            response = client.messages.create(
                model=MODEL_NAME,
                max_tokens=100,
                messages=[{"role": "user", "content": f"Prompt {i+1}: Come up with a name for an alien planet. Respond with a single word."}],
                temperature=temperature
            )
            print(f"Response {i+1}: {response.content[0].text}")

In [ ]:
#demonstrate_temperature()

Temperature is deprecated for many modern Anthropic models. This example demonstrates that the notebook should not rely on the temperature parameter in the same way as older tutorials.

In [24]:
def demonstrate_temperature():
    print("Temperature is deprecated for this model; using the default generation behavior.")
    for i in range(3):
        response = client.messages.create(
            model=MODEL_NAME,
            max_tokens=100,
            messages=[
                {"role": "user", "content": f"Prompt {i+1}: Come up with a name for an alien planet. Respond with a single word."}
            ]
        )
        print(f"Response {i+1}: {response.content[0].text}")

In [25]:
demonstrate_temperature()


Temperature is deprecated for this model; using the default generation behavior.


AttributeError: 'ThinkingBlock' object has no attribute 'text'

This example demonstrates that some older generation controls may no longer apply in the same way, and that the best practice is to write code that is resilient to evolving API behavior.

**TO BE DONE** Provide an alternative code to be used to get the same output as the temperature

## Conclusions  

This notebook introduced the main workflow for working with Anthropic's API: configure the client, send a prompt, process the response, and maintain conversation history. These are core building blocks for many AI applications, including chatbots, assistants, summarizers, and content generation tools.

The most important design principle is that the **Messages API is a structured conversation system**. By sending past messages back to the model, we create context-aware behavior. By controlling output length and stopping conditions, we guide generation more precisely.

As a final best practice, always write defensive code when handling model responses. Not every response block is a plain text block, and robust handling makes this notebook more resilient to future SDK changes and model updates.

## Extra practice

Once you are comfortable with the basics, there are several useful exercises that will help you become more confident with the Anthropic API and with prompt design. These practice ideas are directly related to the topics covered in this notebook and are excellent for building stronger skills.    

This extra practice section is intended to help you move from “following examples” to “building your own AI applications.” The more you practice, the more naturally you will understand how prompt design and interaction patterns affect the model’s output.

### 1. Build a multi-turn chatbot with memory
Try creating a chatbot that remembers the conversation across multiple turns and keeps track of the user’s preferences.

Example idea:  

- Ask the user for their favorite topic    
- Remember it in a list of messages    
- Use that context in future responses    


In [ ]:
messages = [
    {"role": "user", "content": "My favorite topic is space exploration."},
    {"role": "assistant", "content": "Nice! I will remember that."}
]

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=200,
    messages=messages + [
        {"role": "user", "content": "Explain the most important discoveries in astronomy."}
    ]
)

This is a good practice exercise because it helps you understand how context influences the model's answers.

### 2. Compare model behavior with different prompts
Write the same question in different styles and compare the results:

- direct question  
- detailed prompt  
- role-based prompt  
- step-by-step instruction  

Example:

In [ ]:
prompts = [
    "Summarize machine learning in one paragraph.",
    "Explain machine learning to a beginner in simple language.",
    "Act as a teacher and explain machine learning with examples.",
    "Give me a short list of 5 key ideas in machine learning."
]

for prompt in prompts:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=200,
        messages=[{"role": "user", "content": prompt}]
    )
    print("PROMPT:", prompt)
    for block in response.content:
        if hasattr(block, "text"):
            print(block.text)
    print("\n" + "=" * 50 + "\n")

This exercise helps you learn that prompt phrasing strongly affects the quality and style of the output.

### 3. Create a structured output generator
Use stop_sequences and clear formatting instructions to generate tables, lists, or JSON-like data.

In [ ]:
prompt = """
Return a JSON object with:
- name
- purpose
- benefits
- example_use_case

The answer must be valid JSON only.
"""

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[{"role": "user", "content": prompt}]
)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

This is useful when you want the model to produce output that is easy to parse in an application.

### 4. Experiment with different token limits
Try generating the same answer with different max_tokens values and compare the results.

Example:

In [ ]:
for tokens in [50, 150, 300]:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=tokens,
        messages=[{"role": "user", "content": "Write a short explanation of neural networks."}]
    )
    print(f"MAX_TOKENS = {tokens}")
    for block in response.content:
        if hasattr(block, "text"):
            print(block.text)
    print("=" * 40)

This helps you understand how output length changes with the parameter.

### 5. Build a simple assistant with a system-like instruction
Even though this notebook focuses on the Messages API, you can also experiment with prompting patterns that set a persona or behavioral rule.

Example:

In [ ]:
response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=300,
    messages=[
        {"role": "user", "content": "You are a careful research assistant. Explain the difference between AI, ML, and deep learning in 3 bullet points."}
    ]
)
for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

This helps you practice controlling tone, verbosity, and task framing.

### 6. Create a prompt that drafts and improves text
Practice “draft -> improve” workflows by asking the model to first generate a rough answer and then refine it.

Example:

In [ ]:
messages = [
    {"role": "user", "content": "Draft a short paragraph explaining why Python is useful for AI."},
    {"role": "assistant", "content": "Python is widely used in AI because it is simple and has many useful libraries."},
    {"role": "user", "content": "Make it more polished and academic, and keep it under 100 words."}
]

response = client.messages.create(
    model=MODEL_NAME,
    max_tokens=200,
    messages=messages
)

for block in response.content:
    if hasattr(block, "text"):
        print(block.text)

This pattern is commonly used in real applications for writing, editing, and summarization.

### 7. Test edge cases and error handling
A useful practice is to intentionally send edge cases to understand how the API responds:

- empty prompts  
- very long prompts  
- very short outputs  
- ask for unsupported formatting  
- ask for a list with unusual constraints  

Example:  

In [ ]:
try:
    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=50,
        messages=[{"role": "user", "content": ""}]
    )
except Exception as e:
    print("Error:", e)

This improves your ability to write resilient code and handle API errors gracefully.

8. Build a small project
Once these exercises feel natural, you can combine them into a small personal project, for example:

a study buddy assistant
a writing helper
a topic explainer
a question-answer bot
a language tutor
a summarization tool
These projects are a great next step because they combine the concepts from this notebook with real-world use cases.

## Recommended next steps
To continue improving, focus on:

writing clearer prompts
managing conversation memory
controlling output length and structure
handling response blocks safely
learning how to validate and parse outputs
A strong habit is to keep experimenting with small prompts and compare different formulations. This is one of the best ways to understand how language models behave in practice.

